# Tutorial — EnergyScope Pathway Model, with materials

This notebook shows how to run the EnergyScope transition-pathway model **with critical-materials
tracking** programmatically using `run_pathway(..., materials=True, ...)` from `shared.utils`, access the
results, and generate plots.

It follows the same structure as the plain-pathway tutorial (`projects/pathway/docs/tutorial_pathway.ipynb`)
-- read that one first if you haven't already, everything there (basic run, plotting, saving, GWP budget,
myopic mode, ...) still applies unchanged. This notebook only covers what's *added* by `materials=True`.

Make sure you are running this notebook from `projects/critical_materials/` (or that this directory and
the repo root are both on your Python path).

## 1. Import

In [1]:
import sys
sys.path.insert(0, '../..')  # repo root, for `shared.utils`

from shared.utils import run_pathway

## 2. Basic run

The two flags that decide *how* recycling happens (full picture in §7): `materials_recycling_cost=False`
+ `force_max_recycling=True` means no cost signal -- `Recycled_material` is forced instead to the
technical recycling-rate ceiling.

In [2]:
results = run_pathway('test_scenario', materials=True, materials_recycling=True,
                       materials_recycling_cost=False, force_max_recycling=True)

[run_pathway] Window 1/1 done in 691.3s
[run_pathway] Total time: 691.3s
Plotting to: /Users/Paolo/Documents/PdM_code/EnergyScope-Quebec/projects/critical_materials/../../projects/critical_materials/out/test_scenario/graphs
  Plotting [215]: 16_Sankey_2050.html                                    [WARN] GWP 2040: Year_balance=37816.0 kt, allocated=31654.7 kt (diff=-6161.3 kt)
[WARN] GWP 2045: Year_balance=37816.0 kt, allocated=28233.6 kt (diff=-9582.4 kt)
  Plotting [236]: 18_Elec_monthly_2050.html                                 Dashboard: /Users/Paolo/Documents/PdM_code/EnergyScope-Quebec/projects/critical_materials/../../projects/critical_materials/out/test_scenario/graphs/index.html

Done — 236 charts saved to /Users/Paolo/Documents/PdM_code/EnergyScope-Quebec/projects/critical_materials/../../projects/critical_materials/out/test_scenario/graphs
  Plotting [346]: 23c_Material_stock.html                                  Dashboard: /Users/Paolo/Documents/PdM_code/EnergyScope-Quebec/pr

## 3. Exploring results

Same dict of ~30 DataFrames as the plain pathway model, plus material-specific keys.

In [3]:
# Material-specific result keys
[k for k in results if 'material' in k.lower() or 'recycl' in k.lower()]

['limit_material_year',
 'Material_content_year',
 'Decommissioned_material',
 'Recycled_material',
 'Recycled_material_by_process',
 'Disposed_material',
 'Recycling_benefit',
 'Used_recycled_material',
 'Material_stock',
 'C_material',
 'C_material_recycling_tech',
 'Material_content_cumulative',
 'Recycled_material_cumulative',
 'Recycling_benefit_cumulative']

In [ ]:
# Annual gross material demand, per (year, technology, material) [t/year]. Includes the
# pre-2020 legacy fleet (Years == 'YEAR_2020', phase "2015_2020") -- a real quantity, correctly
# computed (material_content_year_calc covers that phase), even though it's not a transition-period
# decision. Dashboard charts filter YEAR_2020 out (Plot_functions._drop_year_2020) for exactly that
# reason, but it's still here if you query results directly.
results['Material_content_year'].head()

### Used vs. banked

Recycled material not needed against a given year's own demand gets banked (`Material_stock`) rather
than wasted, and can be drawn down in a later year (`Used_recycled_material`). Both are indexed by
(Years, Materials) only -- summed across technologies, since recycled material is fungible once
recycled (no "this batch is from 2030" tracking). §7 covers what makes the solver actually draw the
stock down rather than let it accumulate.

In [ ]:
results['Used_recycled_material'].head(), results['Material_stock'].head()

In [6]:
# Annual amount actually recycled, by material, summed across technologies [t/year]
rec = results['Recycled_material']['Recycled_material']
rec[rec.abs() > 1e-9].groupby('Materials').sum().sort_values(ascending=False)

Materials
Fe          1.591711e+06
Al          1.552967e+05
Cu          2.809268e+04
Cr          6.512445e+03
Ni          5.236280e+03
Mn          5.203281e+03
Zn          3.617629e+03
Glass       9.542644e+02
Co          8.754374e+02
Pb          5.482125e+02
Polymers    4.691783e+02
Nd          3.783042e+02
Mo          9.392536e+01
Ag          4.811637e+01
Dy          4.457214e+01
Pr          4.036658e+00
Mg          2.699011e+00
Tb          5.811453e-01
B           5.706593e-01
Te          6.967852e-03
Cd          6.589164e-03
In          1.873348e-03
Si          1.201786e-03
Se          1.170277e-03
Ga          5.952539e-04
Ge          1.989876e-04
Name: Recycled_material, dtype: float64

In [ ]:
# Net demand [t/year], indexed by (Years, Materials): gross demand minus Used_recycled_material --
# exactly the LHS of material_content_year_limit, the real per-year supply constraint (see §8), not a
# reporting-layer reconstruction.
results['Net_demand'].head()

In [ ]:
# Avoided cost from recycling [M$, cumulative over the horizon] only when recycling_cost=True, still to develop
results.get('Recycling_benefit_cumulative')

## 4. Plotting

`build_dashboard=True` (the default when `materials=True`) automatically generates
`out/<case_study>/graphs/index.html` -- a superset of the plain-pathway dashboard, with extra pages for
material demand, decommissioning, and recycling by material/technology.

In [ ]:
results = run_pathway('test_scenario', materials=True, materials_recycling=True,
                       materials_recycling_cost=False, force_max_recycling=True, open_dashboard=True)

## 5. Saving results to disk

Same `save_pkl`/`skip_if_exists` as the plain pathway model -- `materials=True` additionally writes
`_Materials_Results.pkl` alongside `_Results.pkl`.

In [ ]:
results = run_pathway(
    'my_first_materials_run',
    materials=True,
    materials_recycling=True,
    materials_recycling_cost=False,
    force_max_recycling=True,
    description='Baseline materials run',
)  # skip_if_exists=True to reload from disk instead of re-solving, same as the plain model

## 6. The materials pipelines -- where the data comes from

Nothing in `ampl_files/*.dat` is written by hand: `Material_intensity.dat` and `Material_recycling.dat`
are **auto-generated** from the Excel workbooks in `excel_files/`. If you edit an Excel sheet, rerun the
matching pipeline before your next `run_pathway()` call, or the change won't take effect.

| Pipeline | Source Excel | Output | Script |
|---|---|---|---|
| `mi_pipeline` | `Material_intensities.xlsx` | `Material_intensity.dat` | `run_build_mi.py` |
| `rr_pipeline` | `Recycling_rates.xlsx` | `Material_recycling.dat` | `run_build_rr.py` |
| `rt_pipeline` | `Recycling_rates.xlsx` | `Material_recycling_process.dat` | `run_build_rt.py` (only needed for `materials_recycling_process=True`, see §7) |
| `limits_pipeline` | `Material_limits.xlsx` | `Material_limits.dat` | `run_build_limits.py` (only needed for `materials_limit=True`, see §8) |

In [4]:
from run_build_mi import main as build_mi
from run_build_rr import main as build_rr
from run_build_limits import main as build_limits

build_mi()
build_rr()
build_limits()  # only needed if you edited Material_limits.xlsx

[build_table] computed 710 tech intensities in 5.3s
[build_table] built 203770 rows for the Mapping sheet's 710 technologies in 8.9s
[build_table] wrote Material_intensity.dat in 31.3s
[build_table] total: 31.3s

Coverage report:
  not_mapped       : 414
  not_yet_modeled  : 0
  placeholder_zero : 0
  integrated       : 296

not_mapped (414):
  - AL_MAKING
  - AL_MAKING_HR
  - AN_DIG
  - AN_DIG_SI
  - ATR
  - ATR_CCS
  - BATTERY
  - BIOETHANOL_TO_BIOJETFUELS
  - BIOGAS_ATR
  - BIOGAS_ATR_CCS
  - BIOGAS_BIOMETHANE
  - BIOGAS_SMR
  - BIOGAS_SMR_CCS
  - BIOMASS_GAS_EF_H2
  - BIOMASS_GAS_EF_H2_CCS
  - BIOMASS_GAS_FB_H2
  - BIOMASS_GAS_FB_H2_CCS
  - BIOMETHANE_TO_BIOMETHANOL
  - BIOMETHANOL_CARBONYLATION
  - BIOMETHANOL_FT
  - BIOMETHANOL_TO_AROMATICS
  - BIOMETHANOL_TO_OLEFINS
  - BULK_CARRIER
  - BULK_CARRIER_BIO
  - BULK_CARRIER_BIOMEOH
  - BULK_CARRIER_BIOMEOH_ELD
  - BULK_CARRIER_BIO_ELD
  - BULK_CARRIER_ELD
  - BULK_CARRIER_MEOH
  - BULK_CARRIER_MEOH_ELD
  - BUS_FC_H2
  - BUS_FC_H2_SD

,2030,2035,2040,2045,2050
Material limits [t],,,,,
Al,3.212376e+05,4.323270e+05,5.818330e+05,7.830407e+05,1.053829e+06
B,1.687089e+04,2.150297e+04,2.740684e+04,3.493167e+04,4.452253e+04
Cd,1.153858e+01,1.383961e+01,1.659951e+01,1.990978e+01,2.388019e+01
Cr,1.915256e+05,2.663501e+05,3.704066e+05,5.151155e+05,7.163587e+05
Co,1.080000e+03,9.285000e+02,7.770000e+02,1.136403e+03,1.662049e+03
Cu,7.290000e+04,6.135000e+04,4.980000e+04,5.671424e+04,6.458846e+04
Dy,5.634744e+00,7.268964e+00,9.377148e+00,1.209676e+01,1.560513e+01
Ga,4.234171e+00,5.698418e+00,7.669028e+00,1.032111e+01,1.389032e+01
Ge,1.279211e+00,1.534311e+00,1.840284e+00,2.207273e+00,2.647448e+00


## 7. Recycling approaches

`materials_recycling=True` is required for any recycling at all (otherwise `recycling_rate` stays 0
everywhere and everything is disposed). On top of that, two independent `run_pathway` flags control
*how much* gets recycled:

- `materials_recycling_cost` (default `True`): real recycling/disposal costs drive the optimizer's
  choice. Set `False` for no cost signal at all -- in that case `Recycled_material` is otherwise
  solver-indeterminate unless you also set...
- `force_max_recycling` (default `False`): forces `Recycled_material` to exactly the technical ceiling
  (`recycling_rate`).

Once something *is* recycled, using it against that year's own demand vs. banking it for later
(`Used_recycled_material` vs `Material_stock`, previous section) is a separate decision. Without a cost
tied to them, those two are themselves LP-degenerate -- nothing stops the solver from banking material
indefinitely for no reason. `Constraints.mod`'s `material_cost_calc` adds a small banking penalty
(0.001 $/t) on `Material_stock`, charged **every year** the balance stays nonzero (it's a running total,
not a one-off charge) -- so leaving material banked for 20 years costs ~4x more than using it right away,
which is what actually pushes the solver to draw the stock down rather than let it sit.

In [ ]:
results_free = run_pathway('approach_free', materials=True, materials_recycling=True,
                            materials_recycling_cost=False, force_max_recycling=True)
results_cost = run_pathway('approach_real_cost', materials=True, materials_recycling=True,
                            materials_recycling_cost=True, force_max_recycling=False)

## 8. Material production limits

`materials_limit=True` applies manual annual caps from `Material_limits.dat` -- e.g. a Quebec-allocated
share of world neodymium production, so the model can't assume unlimited access to a genuinely scarce
material.

Two separate parameters, only one of which currently has real data. `limit_material_year[y,mat]`
(per-year cap, bounds `Material_content_year - Used_recycled_material` via `material_content_year_limit`)
has real values for 2030-2050 only -- 2020 and 2025 always stay at the large default, since that period's
technology mix is fixed by historical calibration rather than a model decision, and constraining it would
risk artificial infeasibility. `limit_material[mat]` (cumulative, whole-horizon cap) has no data at all
yet, so `material_content_limit` isn't currently binding in any run.

Runs without a real limit (`materials_limit=False`) also take noticeably longer to solve -- without
`limit_material_year`'s tight per-year structure, `Used_recycled_material`/`Material_stock` are only
loosely tied down by the small banking penalty from §7, so Gurobi's root-node cuts take longer to
converge. Expect roughly 3-4x the solve time of an equivalent `materials_limit=True` run.

In [3]:
results_mat_limit = run_pathway('1_baseline_free_limits03', materials=True, materials_recycling=True,
                                 materials_recycling_cost=False, force_max_recycling=True,
                                 materials_limit=True)

[run_pathway] Window 1/1 done in 154.2s
[run_pathway] Total time: 154.2s
Plotting to: /Users/Paolo/Documents/PdM_code/EnergyScope-Quebec/projects/critical_materials/../../projects/critical_materials/out/1_baseline_free_limits03/graphs
  Plotting [214]: 16_Sankey_2050.html                                    [WARN] GWP 2040: Year_balance=37816.0 kt, allocated=33979.0 kt (diff=-3837.0 kt)
[WARN] GWP 2045: Year_balance=37816.0 kt, allocated=31073.3 kt (diff=-6742.7 kt)
  Plotting [235]: 18_Elec_monthly_2050.html                                 Dashboard: /Users/Paolo/Documents/PdM_code/EnergyScope-Quebec/projects/critical_materials/../../projects/critical_materials/out/1_baseline_free_limits03/graphs/index.html

Done — 235 charts saved to /Users/Paolo/Documents/PdM_code/EnergyScope-Quebec/projects/critical_materials/../../projects/critical_materials/out/1_baseline_free_limits03/graphs
  Plotting [355]: 23c_Material_stock.html                                  Dashboard: /Users/Paolo/Documen

In [2]:
results_mat_limit['Material_stock'].head()

NameError: name 'results_mat_limit' is not defined

In [2]:
results_1 = run_pathway('1_baseline_free', materials=True, materials_recycling=True,
                                 materials_recycling_cost=False, force_max_recycling=True,
                                 materials_limit=False)

<function _run_pathway_materials.<locals>._SilentHandler.output at 0x10e75a3e0> returned a result with an exception set


SystemError: <function _run_pathway_materials.<locals>._SilentHandler.output at 0x10e75a3e0> returned a result with an exception set

## 9. Common queries

A few patterns for pulling specific numbers out of a `results` dict without hunting through the
dashboard.

In [ ]:
# Net demand of a single material in a single year [t]
results['Net_demand'].loc[('YEAR_2040', 'Li'), 'Net_demand']

In [ ]:
# One material's full trajectory across years: net demand, used-recycled, banked stock -- side by side
import pandas as pd

material = 'Li'
pd.concat([
    results['Net_demand'].xs(material, level='Materials'),
    results['Used_recycled_material'].xs(material, level='Materials'),
    results['Material_stock'].xs(material, level='Materials'),
], axis=1)

In [ ]:
# Gross demand aggregated across technologies for one material -- drop the mobility SD/MD/LD/ELD
# variants first (see _drop_mob_size_variants' docstring) to avoid double-counting a family tech
# and its variants, same convention MATERIAL_TECHS uses in Constraints.mod.
from Plot_functions import _drop_mob_size_variants

gross = _drop_mob_size_variants(results['Material_content_year']['Material_content_year'])
gross.groupby(['Years', 'Materials']).sum().xs('Li', level='Materials')

In [ ]:
# Materials closest to their per-year supply limit -- only meaningful for a materials_limit=True
# run (results_mat_limit from §8): limit_material_year stays at the large default otherwise.
limit = results_mat_limit['limit_material_year']['limit_material_year']
net = results_mat_limit['Net_demand']['Net_demand']
usage_share = (net / limit).replace([float('inf'), -float('inf')], 0).fillna(0)
usage_share.sort_values(ascending=False).head(10)  # closest to 1.0 = closest to the limit

In [ ]:
# Total recycled per material over the whole horizon -- last year of the _cumulative key is the total
rec_cum = results['Recycled_material_cumulative']['Recycled_material_cumulative']
last_year = rec_cum.index.get_level_values('Years').max()
rec_cum.xs(last_year, level='Years').groupby('Materials').sum().sort_values(ascending=False).head(10)

In [ ]:
# Compare net demand for one material across two runs (results_free / results_cost from §7)
material, year = 'Li', 'YEAR_2040'
{
    'free': results_free['Net_demand'].loc[(year, material), 'Net_demand'],
    'real_cost': results_cost['Net_demand'].loc[(year, material), 'Net_demand'],
}